# Training an Uncertainty Head for `gaokerena-r1.0`

Train a claim-level **uncertainty quantification (UQ) head** on
[`gaokerena/gaokerena-r1.0`](https://huggingface.co/gaokerena/gaokerena-r1.0)
using the Persian medical dataset
[`gaokerena/LUH_Gaokerena_R`](https://huggingface.co/datasets/gaokerena/LUH_Gaokerena_R)
and the [llm-uncertainty-head](https://github.com/IINemo/llm-uncertainty-head) (LUH) framework.

A UQ head is a small auxiliary module that reads the base LLM's attention maps and
token probabilities and predicts, per claim, whether the model is hallucinating.
The base LLM itself stays frozen — only the head is trained.

**Run this notebook on a machine with a CUDA GPU.** It will not work on CPU-only
hardware (the base model is 8B parameters).

| | |
|---|---|
| **GPU** | ~40 GB VRAM recommended. Lower it with a smaller batch size. |
| **Disk** | ~35 GB (16 GB base download + ~16 GB merged copy + checkpoints) |
| **HF token** | Required — the base model `CohereLabs/aya-expanse-8b` is **gated**. Accept its terms on the model page first, with the same account as your token. |

**Steps:** environment check → install → patch known upstream bugs → merge the
LoRA adapter → write config → train → final test score.

## 0. Environment check

Confirm a GPU is visible before downloading ~16 GB.

In [ ]:
import subprocess, sys

print(sys.version)
try:
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                          "--format=csv,noheader"],
                         capture_output=True, text=True, check=True).stdout.strip())
except Exception as e:
    print("nvidia-smi unavailable:", e)
    print("!! This notebook needs a CUDA GPU. Stop here if none is present.")

## 1. Install

Clones the framework and installs dependencies. Skips the clone if the repo is already present.

Every Python call in this notebook uses `sys.executable` — the interpreter running
this kernel — rather than a bare `python`/`pip`. Under `shell=True` the command runs
in `/bin/sh`, which does **not** source your shell profile, so a bare `python` could
resolve to a different interpreter than the kernel (common with conda, or a venv that
is not activated in the subshell). Using `sys.executable` guarantees that what gets
installed here is what the training subprocess later imports.

In [ ]:
import os, subprocess, sys, importlib

REPO_URL = "https://github.com/IINemo/llm-uncertainty-head.git"
REPO_DIR = "llm-uncertainty-head"   # set to "." if this notebook already lives in the repo

def sh(cmd):
    """Run a shell command, raising on failure.

    Always invoke Python as {sys.executable} -- see the note above.
    """
    print("$", cmd)
    r = subprocess.run(cmd, shell=True)
    if r.returncode:
        raise RuntimeError(f"failed ({r.returncode}): {cmd}")

if REPO_DIR != "." and not os.path.isdir(REPO_DIR):
    sh(f"git clone {REPO_URL} {REPO_DIR}")

os.chdir(REPO_DIR)
REPO_ROOT = os.getcwd()
print("repo root  :", REPO_ROOT)
print("interpreter:", sys.executable)

sh(f"{sys.executable} -m pip install -q git+https://github.com/IINemo/lm-polygraph.git@dev")
sh(f"{sys.executable} -m pip install -q -e .")
sh(f"{sys.executable} -m pip install -q peft")     # needed for the LoRA merge

# Packages installed after the kernel started may be missed by cached import paths
# (notably the .pth file that `pip install -e .` drops in). Refresh the finders so
# the imports in later cells resolve without a kernel restart.
importlib.invalidate_caches()

print("\ninstall complete")
for m in ("transformers", "datasets", "huggingface_hub", "peft", "lm_polygraph", "torch"):
    try:
        mod = importlib.import_module(m)
        print(f"  {m:16} {getattr(mod, '__version__', '?')}")
    except Exception as e:
        print(f"  {m:16} !! {type(e).__name__}: {e}")
        print(f"                   -> restart the kernel, then re-run from the cell above "
              f"(the pip lines are no-ops the second time)")

## 2. Hugging Face token

The base model is gated, so a token is required. Accept the terms at
<https://huggingface.co/CohereLabs/aya-expanse-8b> with the same account first,
otherwise the download returns **403**.

In [ ]:
import os, getpass
from huggingface_hub import HfApi

HF_TOKEN = os.environ.get("HF_TOKEN") or getpass.getpass("HF token (input hidden): ").strip()
os.environ["HF_TOKEN"] = HF_TOKEN

api = HfApi(token=HF_TOKEN)
print("logged in as:", api.whoami().get("name"))

# fail early and clearly if gated access has not been granted
try:
    api.model_info("CohereLabs/aya-expanse-8b")
    print("base model access: OK")
except Exception as e:
    print("!! base model NOT accessible:", type(e).__name__)
    print("   Accept the licence at https://huggingface.co/CohereLabs/aya-expanse-8b")

## 3. Patch two upstream bugs

Both are silent — they produce wrong numbers rather than errors — so patch before training.

**(a) `apply_chat_template` returns a `BatchEncoding`.** On newer `transformers`,
`tokenizer.apply_chat_template(..., tokenize=True)` returns a mapping, not a list.
Code that calls `len()` on it gets **2** (the number of dict keys) instead of the
token count. In `run_train_uhead.py` that corrupts the reply boundary:

```python
reply_len = len(examples[i]['input_ids']) - len(examples[i]['prompt_tokens'])
```

`context_lengths` ends up ~17 tokens too small, so **claim masks land on prompt
tokens instead of the reply** — training on the wrong positions, with no error raised.

**(b) `torch_dtype` has no effect.** `load_model()` does
`globals().get("torch.float16")`, which is always `None`, so the base model loads
in fp32 (~32 GB for 8B) whatever the YAML says. This is the most likely cause of OOM.

The cell below applies both fixes idempotently — safe to re-run.

In [ ]:
import re
from pathlib import Path

def patch(path, old, new, label):
    p = Path(path); src = p.read_text()
    if new in src:
        print(f"  [already patched] {label}"); return
    if old not in src:
        print(f"  [PATTERN NOT FOUND -> check manually] {label}"); return
    p.write_text(src.replace(old, new, 1))
    print(f"  [patched] {label}")

# --- (a) helper in luh/utils.py -----------------------------------------
u = Path("luh/utils.py"); src = u.read_text()
if "def chat_template_ids" not in src:
    helper = "\n".join([
        "",
        "",
        "def chat_template_ids(tokenizer, conversation, **kwargs):",
        "    # apply_chat_template(tokenize=True) returning plain token ids.",
        "    # Newer transformers returns a BatchEncoding mapping; len() of that is",
        "    # its key count (2), not the token count, which silently corrupts the",
        "    # reply-boundary arithmetic in the collators.",
        "    out = tokenizer.apply_chat_template(conversation, tokenize=True, **kwargs)",
        "    if isinstance(out, Mapping) or hasattr(out, 'input_ids'):",
        "        out = out['input_ids']",
        "    return out",
        "",
    ])
    marker = "def load_feature_extractor(config, base_model):\n    return luh.feature_extractors.combined.load_extractor(config, base_model)\n"
    src = src.replace(marker, marker + helper, 1)
    u.write_text(src)
    print("  [patched] luh/utils.py: added chat_template_ids")
else:
    print("  [already patched] luh/utils.py: chat_template_ids")

patch("luh/cli/train/run_train_uhead.py",
      "from luh.utils import load_any_dataset\n",
      "from luh.utils import load_any_dataset, chat_template_ids\n",
      "run_train_uhead.py: import")

patch("luh/cli/train/run_train_uhead.py",
      '        return {"prompt_tokens": tokenizer.apply_chat_template([{"role": "user", "content": inst["question"]}], add_generation_prompt=True)}',
      '        return {"prompt_tokens": chat_template_ids(\n'
      '            tokenizer,\n'
      '            [{"role": "user", "content": inst["question"]}],\n'
      '            add_generation_prompt=True,\n'
      '        )}',
      "run_train_uhead.py: prompt_tokens (context_lengths fix)")

# --- (b) torch_dtype ----------------------------------------------------
patch("luh/cli/train/run_train_uhead.py",
      "    config.model.torch_dtype = globals().get(config.model.torch_dtype)",
      '    config.model.torch_dtype = getattr(\n'
      '        torch, str(config.model.torch_dtype).split(".")[-1], None)',
      "run_train_uhead.py: torch_dtype resolution")

print("\npatching done")

## 4. Why the adapter cannot be used directly

`gaokerena/gaokerena-r1.0` is a **LoRA adapter**, not a standalone model. Its repo has
`adapter_config.json` and `adapter_model.safetensors` — but **no `config.json` and no full weights**.

The LUH trainer loads the base model with a plain call in `load_model()`:

```python
base_model = AutoModelForCausalLM.from_pretrained(
    config.model.pretrained_model_name_or_path, ...)
```

There is **no PEFT handling in that path**, so pointing the config at the adapter fails.
The fix is to merge the adapter into its base once and train against the merged model.

The next cell confirms this and reads the base model name from the adapter config.

In [ ]:
import json
from huggingface_hub import hf_hub_download

ADAPTER = "gaokerena/gaokerena-r1.0"
info = api.model_info(ADAPTER)
files = sorted(f.rfilename for f in info.siblings)
print("files in the adapter repo:")
for f in files:
    print("   ", f)
print("\nhas config.json (full model)? ->", "config.json" in files)

acfg = json.load(open(hf_hub_download(ADAPTER, "adapter_config.json", token=HF_TOKEN)))
print("\npeft_type   :", acfg.get("peft_type"))
print("base model  :", acfg.get("base_model_name_or_path"))
print("lora rank r :", acfg.get("r"), " alpha:", acfg.get("lora_alpha"))

# CohereForAI was renamed to CohereLabs; same repo.
BASE = "CohereLabs/aya-expanse-8b"
print("\nwill merge into:", BASE)

### Tokenizer compatibility

The dataset's `input_ids` were produced with the `aya-expanse-8b` tokenizer. If the
adapter shipped a different tokenizer, every stored `input_ids` would be invalid and
the dataset would need regenerating. The next cell verifies they match — it downloads
only tokenizer files, not weights.

In [ ]:
from transformers import AutoTokenizer
from datasets import load_dataset

tok_adapter = AutoTokenizer.from_pretrained(ADAPTER, token=HF_TOKEN)
tok_base    = AutoTokenizer.from_pretrained(BASE, token=HF_TOKEN)

print("vocab sizes:", len(tok_adapter), len(tok_base), "| equal:", len(tok_adapter) == len(tok_base))
print("special ids identical:", sorted(tok_adapter.all_special_ids) == sorted(tok_base.all_special_ids))
print("eos:", repr(tok_adapter.eos_token), tok_adapter.eos_token_id)

ds_check = load_dataset("gaokerena/LUH_Gaokerena_R", split="test")
same = 0
for i in range(len(ds_check)):
    q = ds_check[i]["question"]
    a = list(tok_base.apply_chat_template([{"role": "user", "content": q}],
                                          tokenize=True, add_generation_prompt=True)["input_ids"])
    g = list(tok_adapter.apply_chat_template([{"role": "user", "content": q}],
                                             tokenize=True, add_generation_prompt=True)["input_ids"])
    if a == g == list(ds_check[i]["input_ids"])[:len(a)]:
        same += 1
print(f"\nchat-template tokenization matches stored input_ids: {same}/{len(ds_check)} rows")
assert same == len(ds_check), "tokenizer mismatch - the dataset would need regenerating"
print("OK - the dataset's input_ids are valid for this model")

## 5. Merge the LoRA adapter

Loads the base model, applies the adapter, merges the LoRA weights in, and saves a
standalone model. Roughly 10–30 minutes, mostly download time.

**Memory note.** This holds the full ~16 GB model in the kernel. Training (§7) loads
its *own* copy in a subprocess, so the kernel's copy must be released first or the
GPU will run out of memory. The cell explicitly deletes every reference and empties
the CUDA cache at the end, and the next cell verifies the memory actually came back.

Set `MERGE_DEVICE = "cpu"` to merge on CPU instead: slower, but it never touches VRAM,
so there is no contention with training at all. Needs ~20 GB of host RAM.

In [ ]:
import gc, os
import torch
from transformers import AutoModelForCausalLM
from peft import PeftModel

MERGED = "./workdir/gaokerena-r1.0-merged"
MERGE_DEVICE = "auto"      # "auto" = use the GPU; "cpu" = keep VRAM completely free
MERGE_DTYPE = torch.float16

def vram_gb():
    return torch.cuda.memory_allocated() / 1e9 if torch.cuda.is_available() else 0.0

print(f"VRAM before: {vram_gb():.2f} GB")

print(f"\nloading base {BASE} ...")
base = AutoModelForCausalLM.from_pretrained(
    BASE, torch_dtype=MERGE_DTYPE, device_map=MERGE_DEVICE,
    token=HF_TOKEN, trust_remote_code=True, low_cpu_mem_usage=True)

print(f"applying adapter {ADAPTER} ...")
merged = PeftModel.from_pretrained(base, ADAPTER, token=HF_TOKEN)

print("merging LoRA weights into the base ...")
merged = merged.merge_and_unload()

print(f"saving -> {MERGED}")
os.makedirs(os.path.dirname(MERGED) or ".", exist_ok=True)
merged.save_pretrained(MERGED, safe_serialization=True)

# the tokenizer the model was fine-tuned with (verified identical to the base in §4)
tok_adapter.save_pretrained(MERGED)

# Release the model before training loads its own copy. merge_and_unload() returns a
# new object but `base` still references the same underlying modules, so BOTH names
# must go. Nothing above is left as the cell's return value, which would otherwise be
# retained by Jupyter in Out[] / _ and keep the memory alive.
del merged, base
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print(f"\nVRAM after cleanup: {vram_gb():.2f} GB")
print("saved files:", sorted(os.listdir(MERGED))[:10])

In [ ]:
# Verify the merge produced a loadable standalone model and that memory was released.
import json, os

need = {"config.json"}
have = set(os.listdir(MERGED))
weights = [f for f in have if f.endswith(".safetensors")]

print("config.json present :", bool(need & have))
print("weight shards       :", len(weights))
print("tokenizer present   :", any(f.startswith("tokenizer") for f in have))

cfg = json.load(open(os.path.join(MERGED, "config.json")))
print("architectures       :", cfg.get("architectures"))
print("vocab_size          :", cfg.get("vocab_size"))

free = vram_gb()
print(f"\nVRAM still held by this kernel: {free:.2f} GB")
if free > 1.0:
    print("!! Memory was not fully released. Restart the kernel before training,")
    print("   then re-run cells 1-2 only (skip the merge - it is already saved).")
else:
    print("OK - the GPU is free for training.")

## 6. Training config

Differences from the upstream `configs/run_train_uhead.yaml`:

| setting | value | why |
|---|---|---|
| `model.pretrained_model_name_or_path` | merged dir | the adapter cannot be loaded directly |
| `dataset.path` | `hf:gaokerena/LUH_Gaokerena_R` | `hf:` pulls from the Hub |
| `dataset.validation` | `eval` | keeps the 50-row `test` split unseen for a final unbiased score |
| `ue_layer.pos_weight` | `4.53` | measured class ratio on this dataset's `train` split (39,455 true : 8,714 hallucinated); upstream `3` was tuned for a different dataset |
| `per_device_train_batch_size` | `8` | upstream `32` is very heavy with an 8B base and full-layer attention output |

Raise the batch size if your GPU has room; lower it to 4 or 2 on OOM.

In [ ]:
CONFIG_PATH = "configs/run_train_uhead_gaokerena_r.yaml"
OUTPUT_DIR = "./workdir/train_gaokerena_r"
BATCH_SIZE = 8

config_yaml = f"""
hydra:
  run:
    dir: ${{output_dir}}/${{now:%Y-%m-%d}}/${{now:%H-%M-%S}}

model:
  pretrained_model_name_or_path: {MERGED}
  device_map: cuda
  trust_remote_code: true
  torch_dtype: torch.float16

ue_layer:
  path: ''
  pos_weight: 4.53
  output_attention: true
  head_cfg:
    head_type: claim
    feature_extractor:
    - name: luh.feature_extractors.basic_attention
      layer_nums: all
      attn_history_sz: 3
      pool: false
    - name: luh.feature_extractors.token_probabilities
      top_n: 4
    uncertainty_head:
      head_dim: 768
      n_layers: 2
      n_heads: 8
      dropout: 0.1

dataset:
  path: hf:gaokerena/LUH_Gaokerena_R
  num_instances: 0
  test_size: 0.0
  validation: eval

training_arguments:
  num_train_epochs: 10
  learning_rate: 0.0001
  warmup_ratio: 0.05
  weight_decay: 0.1
  gradient_accumulation_steps: 1
  per_device_train_batch_size: {BATCH_SIZE}
  max_grad_norm: 1.0

do_train: true
do_eval: true
do_hyperopt: false
do_save_checkpoints: true
do_save_final_model: true
report_to: none
output_dir: {OUTPUT_DIR}
do_predict: false
"""

with open(CONFIG_PATH, "w") as f:
    f.write(config_yaml.lstrip())
print(f"wrote {CONFIG_PATH}")
print(config_yaml)

## 7. Train

The base LLM is frozen; only the UQ head is trained (`param.requires_grad = "ue_head" in name`).

Per-epoch metrics are reported on the **`eval`** split; the best checkpoint is chosen on
**`pr_auc`**, with early stopping after 3 epochs without improvement.

This is the long step — expect hours. Output goes to `./workdir/train_gaokerena_r/<date>/<time>/`.

In [ ]:
sh(f"CUDA_VISIBLE_DEVICES=0 {sys.executable} -m luh.cli.train.run_train_uhead "
   f"--config-dir=./configs --config-name=run_train_uhead_gaokerena_r.yaml")

In [ ]:
# locate the run directory just produced and the saved head
import glob, os

runs = sorted(glob.glob(os.path.join(OUTPUT_DIR, "*", "*")), key=os.path.getmtime)
assert runs, f"no run directories under {OUTPUT_DIR}"
LATEST_RUN = runs[-1]
HEAD_PATH = os.path.join(LATEST_RUN, "model")

print("latest run :", LATEST_RUN)
print("saved head :", HEAD_PATH, "| exists:", os.path.isdir(HEAD_PATH))
print("contents   :", sorted(os.listdir(LATEST_RUN)))

## 8. Final score on the held-out `test` split

**The framework uses only ONE held-out split per run.** In `load_data()`:

```python
test_dataset = dataset[config.dataset.validation]
tokenized_data = DatasetDict({"train": ..., "test": test_dataset})
```

Whatever `dataset.validation` names is re-keyed internally as `"test"` and used for
*everything*: `eval_dataset`, per-epoch metrics, `metric_for_best_model`, early
stopping and `trainer.predict()`. **Any other split is never loaded.**

Hence two runs:

| run | `dataset.validation` | split | purpose |
|---|---|---|---|
| training (above) | `eval` | 92 rows | model selection / early stopping |
| this cell | `test` | 50 rows | final unbiased score |

Scoring on `test` during training would leak, since it would drive checkpoint selection.

`ue_layer.path` loads the trained head instead of initialising a new one, and
`do_train=false` runs evaluation only.

In [ ]:
sh(f"CUDA_VISIBLE_DEVICES=0 {sys.executable} -m luh.cli.train.run_train_uhead "
   f"--config-dir=./configs --config-name=run_train_uhead_gaokerena_r.yaml "
   f"dataset.validation=test "
   f"ue_layer.path={HEAD_PATH} "
   f"do_train=false do_eval=true")

## 9. Notes

**Metrics.** For a claim-level head (`head_cfg.head_type: claim`), `compute_metrics_claims`
reports `accuracy`, `precision`, `recall`, `f1`, `roc_auc`, `pr_auc`. A token-level head
additionally reports `neg_f1`, `neg_roc_auc`, `neg_pr_auc` and `claim_level_pr_auc`.

**Out of memory?** In order of effect:
1. lower `BATCH_SIZE` in the config cell (8 → 4 → 2)
2. raise `gradient_accumulation_steps` to keep the effective batch size
3. confirm the `torch_dtype` patch in §3 applied — without it the base model loads in
   fp32 (~32 GB for 8B) regardless of the YAML

**Publishing the head.** After training:

```python
from huggingface_hub import HfApi
HfApi(token=HF_TOKEN).upload_folder(
    folder_path=HEAD_PATH, repo_id="<org>/<head-name>", repo_type="model")
```

**Third known upstream bug** (does not affect training, only dataset regeneration):
`get_prompts()` in `run_extract_verify_claims.py` indexes prompt dicts by *module* name,
raising a `KeyError` that a bare `except: continue` swallows — so custom per-language
prompts are silently ignored.